# A2A ToolSpec - Agent-to-Agent Protocol Integration

This notebook demonstrates how to use the A2A (Agent-to-Agent) protocol with LlamaIndex.

## What is A2A?

A2A (Agent-to-Agent) is a protocol by Google that enables agents to discover and communicate with each other using a standardized JSON-RPC interface over HTTP. It provides:

- **AgentCard**: A standard discovery mechanism (at `/.well-known/agent-card.json`) that describes an agent's capabilities
- **Skills**: Discrete capabilities an agent can perform, each with an input schema
- **Tasks**: Asynchronous execution model with polling for results
- **Streaming**: Server-Sent Events (SSE) for real-time updates

## Two Ways to Use A2A with LlamaIndex

1. **Outbound (Client)**: Connect to a remote A2A agent and use its skills as LlamaIndex FunctionTools
2. **Inbound (Server)**: Expose your LlamaIndex Workflow as an A2A agent that others can discover and call

This notebook covers both patterns.

In [ ]:
# Install dependencies
%pip install llama-index-tools-a2a llama-index-core llama-index-llms-openai httpx uvicorn

---

## Part 1: Inbound - Expose a Workflow as an A2A Agent

First, let's create a simple LlamaIndex Workflow and expose it as an A2A-compatible agent using `workflow_as_a2a()`.

In [ ]:
from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step
from llama_index.tools.a2a import workflow_as_a2a

class ResearchStart(StartEvent):
    topic: str
    depth: int = 2

class ResearchWorkflow(Workflow):
    @step
    async def research(self, start_event: ResearchStart) -> StopEvent:
        # Simulate research work
        result = f"Research on '{start_event.topic}' completed at depth {start_event.depth}. "
        result += "Key findings: A2A enables agent interoperability. "
        result += "MCP provides tool access. Both complement each other."
        return StopEvent(result=result)

# Create the A2A-compatible FastAPI app
app = workflow_as_a2a(ResearchWorkflow(), url="http://localhost:8000")

# Inspect the AgentCard
import json
card = app.state.agent_card
print(json.dumps(card.model_dump(mode='json'), indent=2))

The app is now an A2A agent! It exposes:

- `GET /.well-known/agent-card.json` - Discovery endpoint
- `POST /` - JSON-RPC endpoint for `message/send`, `tasks/get`, etc.

Run it with uvicorn in a separate terminal:

```bash
uvicorn examples.a2a_server:app --host 0.0.0.0 --port 8000
```

Or run it programmatically for testing (which we'll do below).

In [ ]:
# Test the inbound server programmatically
from httpx import AsyncClient, ASGITransport
import asyncio
import json

async def test_inbound_server():
    transport = ASGITransport(app=app)
    async with AsyncClient(transport=transport, base_url="http://test") as client:
        # 1. Get AgentCard
        r = await client.get("/.well-known/agent-card.json")
        print("=== AgentCard ===")
        print(json.dumps(r.json(), indent=2))
        print()
        
        # 2. Send a message (message/send)
        resp = await client.post("/", json={
            "jsonrpc": "2.0",
            "method": "message/send",
            "params": {
                "message": {
                    "messageId": "msg-1",
                    "role": "user",
                    "parts": [{"kind": "text", "text": '{"topic": "AI agents", "depth": 3}'}]
                }
            },
            "id": 1
        })
        print("=== message/send response ===")
        result = resp.json()["result"]
        print(f"Task ID: {result['id']}")
        print(f"Status: {result['status']['state']}")
        print(f"Artifacts: {result['artifacts']}")
        print()
        
        # 3. Get the task (tasks/get)
        task_id = result["id"]
        get_resp = await client.post("/", json={
            "jsonrpc": "2.0",
            "method": "tasks/get",
            "params": {"id": task_id},
            "id": 2
        })
        print("=== tasks/get response ===")
        print(json.dumps(get_resp.json(), indent=2))

asyncio.run(test_inbound_server())

---

## Part 2: Outbound - Connect to an A2A Agent as Tools

Now let's connect to an A2A agent (like the one we just created) and use its skills as LlamaIndex FunctionTools.

In [ ]:
# Start the server in background for testing
import uvicorn
from multiprocessing import Process
import time

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_process = Process(target=run_server)
server_process.start()
time.sleep(2)  # Wait for server to start
print("Server started at http://127.0.0.1:8000")

In [ ]:
# Now connect as a client
from llama_index.tools.a2a import (
    BasicA2AClient,
    A2AToolSpec,
    aget_tools_from_a2a_card,
    get_tools_from_a2a_card
)

# Method 1: Using BasicA2AClient directly
client = BasicA2AClient(
    agent_card_url="http://127.0.0.1:8000/.well-known/agent-card.json",
    timeout=30
)

# Discover skills
card = await client.get_agent_card()
print(f"Agent: {card.name}")
print(f"Description: {card.description}")
print(f"Skills:")
for skill in card.skills:
    print(f"  - {skill.name}: {skill.description}")
    print(f"    Input schema: {skill.input_schema}")

In [ ]:
# Method 2: Convert to LlamaIndex FunctionTools using A2AToolSpec
tool_spec = A2AToolSpec(client)
tools = await tool_spec.to_tool_list_async()

print(f"Created {len(tools)} FunctionTool(s)")
for tool in tools:
    print(f"  - {tool.metadata.name}: {tool.metadata.description}")

In [ ]:
# Method 3: Convenience function (simplest)
tools = await aget_tools_from_a2a_card(
    "http://127.0.0.1:8000/.well-known/agent-card.json"
)

print(f"Created {len(tools)} FunctionTool(s) via convenience function")
for tool in tools:
    print(f"  - {tool.metadata.name}: {tool.metadata.description}")

---

## Part 3: Using A2A Tools in a LlamaIndex Agent

Now let's use the discovered A2A tools in a LlamaIndex agent (ReActAgent or FunctionAgent).

In [ ]:
# Set up an LLM (using OpenAI as example)
import os
from llama_index.llms.openai import OpenAI

# You need to set OPENAI_API_KEY environment variable
# os.environ["OPENAI_API_KEY"] = "your-key-here"

llm = OpenAI(model="gpt-4o-mini")

In [ ]:
# Create a ReActAgent with the A2A tools
from llama_index.core.agent import ReActAgent

agent = ReActAgent.from_tools(
    tools,
    llm=llm,
    verbose=True
)

In [ ]:
# Run the agent - it will use the A2A skill!
response = await agent.achat("Research the topic 'multi-agent systems' with depth 3")
print(response)

---

## Part 4: Advanced - Filtering and Partial Parameters

A2AToolSpec supports filtering specific skills and pre-filling parameters.

In [ ]:
# Filter to only specific skills
client2 = BasicA2AClient(
    agent_card_url="http://127.0.0.1:8000/.well-known/agent-card.json"
)
tool_spec_filtered = A2AToolSpec(
    client2,
    allowed_tools=["researchworkflow"]  # Only include this skill
)
tools_filtered = await tool_spec_filtered.to_tool_list_async()
print(f"Filtered tools: {[t.metadata.name for t in tools_filtered]}")

In [ ]:
# Pre-fill parameters globally
client3 = BasicA2AClient(
    agent_card_url="http://127.0.0.1:8000/.well-known/agent-card.json"
)
tool_spec_partial = A2AToolSpec(
    client3,
    global_partial_params={"depth": 5}  # Default depth for all skills
)
tools_partial = await tool_spec_partial.to_tool_list_async()
print(f"Tools with global partial params: {[t.metadata.name for t in tools_partial]}")

In [ ]:
# Clean up
server_process.terminate()
server_process.join()
print("Server stopped")

---

## Part 5: Complete End-to-End Example

Here's a complete script you can run that demonstrates the full A2A flow:

1. Start an A2A server (workflow_as_a2a)
2. Connect a client and discover skills
3. Convert skills to FunctionTools
4. Use in a LlamaIndex agent

In [ ]:
# Complete example as a standalone script
complete_example = '''
import asyncio
from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step
from llama_index.tools.a2a import workflow_as_a2a, aget_tools_from_a2a_card
from llama_index.core.agent import ReActAgent
from llama_index.llms.openai import OpenAI
import uvicorn
from multiprocessing import Process
import time
import httpx

class ResearchStart(StartEvent):
    topic: str
    depth: int = 2

class ResearchWorkflow(Workflow):
    @step
    async def research(self, start_event: ResearchStart) -> StopEvent:
        return StopEvent(result=f"Research on '{start_event.topic}' completed at depth {start_event.depth}.")

async def main():
    # 1. Create and start A2A server
    app = workflow_as_a2a(ResearchWorkflow(), url="http://localhost:8000")
    
    def run_server():
        uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")
    
    server = Process(target=run_server)
    server.start()
    time.sleep(2)
    
    try:
        # 2. Connect client and get tools
        tools = await aget_tools_from_a2a_card(
            "http://127.0.0.1:8000/.well-known/agent-card.json"
        )
        print(f"Discovered {len(tools)} tools: {[t.metadata.name for t in tools]}")
        
        # 3. Use in agent
        llm = OpenAI(model="gpt-4o-mini")
        agent = ReActAgent.from_tools(tools, llm=llm, verbose=True)
        
        response = await agent.achat("Research AI agents with depth 3")
        print(f"Agent response: {response}")
    finally:
        server.terminate()
        server.join()

if __name__ == "__main__":
    asyncio.run(main())
'''
print(complete_example)

---

## Part 6: Architecture Comparison (MCP vs A2A)

| Concept | MCP | A2A |
|---------|-----|-----|
| Discovery | `list_tools()` | `GET /.well-known/agent-card.json` |
| Invocation | `call_tool(name, args)` | `message/send` + `tasks/get` polling |
| Protocol | JSON-RPC (custom) | JSON-RPC 2.0 (standardized) |
| Transport | stdio / SSE | HTTP / SSE |
| Inbound Server | `MCPServer` | `workflow_as_a2a()` → FastAPI app |
| Outbound Client | `BasicMCPClient` | `BasicA2AClient` |
| Tool Spec | `McpToolSpec` | `A2AToolSpec` |
| Workflow Wrapper | `workflow_as_mcp()` | `workflow_as_a2a()` |

Both integrations follow the same pattern in LlamaIndex, making it easy to switch between or use both protocols.

---

## Summary

You now know how to:

1. **Expose** a LlamaIndex Workflow as an A2A agent using `workflow_as_a2a()`
2. **Discover** remote A2A agents via their AgentCard
3. **Convert** A2A skills to LlamaIndex FunctionTools using `A2AToolSpec` or `aget_tools_from_a2a_card()`
4. **Use** A2A tools in LlamaIndex agents (ReActAgent, FunctionAgent, AgentWorkflow)
5. **Filter** skills and **pre-fill** parameters for convenience

This enables true agent-to-agent interoperability - your LlamaIndex agents can now seamlessly call other A2A-compatible agents (LangGraph, CrewAI, custom agents, etc.) and be called by them!